# VectorForge on SIFT1M

Every VectorForge index (and its Faiss counterpart) on **1,000,000** SIFT vectors, where the
approximate indexes are supposed to beat exact search. On SIFT10K, exact search won almost
everything; this notebook checks whether that flips at 100x the size.

**How to run it on Kaggle**
1. *Settings -> Internet -> On* (needs a phone-verified Kaggle account). It downloads the
   dataset and installs VectorForge from GitHub.
2. Use a **CPU** session (no GPU needed). It needs about 15 GB of RAM; Kaggle CPU sessions have about 30 GB.
3. *Save Version -> Save & Run All (Commit)*. It then runs in the background (expect
   **1.5-3 hours**: building HNSW one vector at a time in Python is the slow part) and you can close the tab.
4. When it's done, open the version, go to **Output**, and download
   `sift1m_results.csv` and `sift1m.png`. Put them in `docs/` in the repo.

Results are saved after every index, so a crash late in the run doesn't lose the earlier numbers.

In [ ]:
# --- setup: Faiss, and VectorForge straight from GitHub
!pip install -q faiss-cpu
!pip install -q --ignore-requires-python git+https://github.com/DHaKK-SHeSH/vectorforge.git

In [ ]:
# --- settings
N_QUERIES = 1000                  # of the 10,000 SIFT1M queries (enough for a stable recall@10)
K = 10
BATCH = 100                       # queries per search call (keeps the (queries x 1M) distance matrices small)
TRAIN_N = 100_000                 # training vectors for k-means / PQ codebooks

NLIST = 1024                      # IVF / IVF-PQ cells (about 1,000 vectors per cell)
NPROBES = [1, 4, 16, 64]
PQ_M = 8                          # 8 bytes per vector
HNSW_M = 16
HNSW_EF_CONSTRUCTION = 64         # 100 on SIFT10K; lower here to keep the 1M build time reasonable
EF_SEARCHES = [16, 32, 64, 128]

OUT_DIR = "/kaggle/working" 

In [ ]:
# --- data: SIFT1M from the TEXMEX corpus (FTP), or the ann-benchmarks copy (HTTP) as a fallback
import os, tarfile, time, urllib.request
import numpy as np
from vectorforge.io import read_fvecs, read_ivecs

def load_sift1m():
    try:
        if not os.path.exists("sift/sift_base.fvecs"):
            print("downloading sift.tar.gz (160 MB) from ftp.irisa.fr ...")
            urllib.request.urlretrieve("ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz", "sift.tar.gz")
            with tarfile.open("sift.tar.gz") as t:
                t.extractall(".")
        xb = read_fvecs("sift/sift_base.fvecs")
        xt = read_fvecs("sift/sift_learn.fvecs")
        xq = read_fvecs("sift/sift_query.fvecs")
        gt = read_ivecs("sift/sift_groundtruth.ivecs")
        return xb, xt, xq, gt, "TEXMEX (separate 100k learn set)"
    except Exception as e:
        print("FTP failed:", repr(e), "-> trying ann-benchmarks")
    import h5py
    if not os.path.exists("sift-128-euclidean.hdf5"):
        print("downloading sift-128-euclidean.hdf5 (500 MB) ...")
        urllib.request.urlretrieve("http://ann-benchmarks.com/sift-128-euclidean.hdf5", "sift-128-euclidean.hdf5")
    with h5py.File("sift-128-euclidean.hdf5", "r") as f:
        xb = f["train"][:].astype(np.float32)
        xq = f["test"][:].astype(np.float32)
        gt = f["neighbors"][:].astype(np.int64)
    idx = np.random.default_rng(0).choice(len(xb), TRAIN_N, replace=False)
    return xb, xb[idx], xq, gt, "ann-benchmarks (learn set = 100k sampled base vectors)"

xb, xt, xq, gt, source = load_sift1m()
xt = xt[:TRAIN_N]
xq, gt = xq[:N_QUERIES], gt[:N_QUERIES, :K]
print(f"source: {source}")
print(f"base {xb.shape}  train {xt.shape}  queries {xq.shape}  ground truth {gt.shape}")

In [ ]:
# --- helpers: timing, recall, memory, saving
import csv
import faiss
from vectorforge.metrics import recall_at_k

faiss.omp_set_num_threads(1)      # all SEARCH timings single-threaded (fair vs NumPy loops)
N_CPU = os.cpu_count()
rows = []                         # (index, impl, setting, recall, qps, memory_mb, build_s)

def batched_search(index, q, k):
    ids = [index.search(q[s:s + BATCH], k)[1] for s in range(0, len(q), BATCH)]
    return np.vstack(ids)

def measure(index, k=K):
    t0 = time.perf_counter()
    ids = batched_search(index, xq, k)
    return recall_at_k(ids, gt, k), len(xq) / (time.perf_counter() - t0)

def build(fn):
    """Run fn() and return (its result, seconds)."""
    t0 = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t0

def faiss_build(fn):
    """Faiss builds use all cores (only searches are timed single-threaded)."""
    faiss.omp_set_num_threads(N_CPU)
    try:
        return build(fn)
    finally:
        faiss.omp_set_num_threads(1)

def faiss_mb(index):
    return faiss.serialize_index(index).nbytes / 1e6

def record(name, impl, setting, rec, qps, mb, build_s):
    rows.append((name, impl, setting, rec, qps, mb, build_s))
    print(f"{name:<7} {impl:<12} {setting:<16} recall@10={rec:.3f}  QPS={qps:>9,.0f}  "
          f"memory={mb:8.1f} MB  build={build_s:6.1f} s")
    with open(f"{OUT_DIR}/sift1m_results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["index", "impl", "setting", "recall_at_10", "qps", "memory_mb", "build_s"])
        for r in rows:
            w.writerow([r[0], r[1], r[2], f"{r[3]:.4f}", f"{r[4]:.1f}", f"{r[5]:.1f}", f"{r[6]:.1f}"])

MB = 1e6
n, d = xb.shape

In [ ]:
# --- 1. Flat (exact search): the reference
from vectorforge.flat import FlatIndex

flat, b = build(lambda: (lambda i: (i.add(xb), i)[1])(FlatIndex("l2")))
rec, qps = measure(flat)
record("Flat", "VectorForge", "-", rec, qps, xb.nbytes / MB, b)

fflat, b = faiss_build(lambda: (lambda i: (i.add(xb), i)[1])(faiss.IndexFlatL2(d)))
rec, qps = measure(fflat)
record("Flat", "Faiss", "-", rec, qps, faiss_mb(fflat), b)
del flat, fflat

In [ ]:
# --- 2. IVF
from vectorforge.ivf import IVFIndex

def build_ivf():
    index = IVFIndex(nlist=NLIST)
    index.train(xt)
    index.add(xb)
    return index

ivf, b = build(build_ivf)
ivf_mb = (xb.nbytes + 8 * n + ivf.centroids.nbytes) / MB          # vectors + ids + centroids
for p in NPROBES:
    ivf.nprobe = p
    rec, qps = measure(ivf)
    record("IVF", "VectorForge", f"nprobe={p}", rec, qps, ivf_mb, b)
del ivf

def build_faiss_ivf():
    index = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, NLIST)
    index.train(xt)
    index.add(xb)
    return index

fivf, b = faiss_build(build_faiss_ivf)
for p in NPROBES:
    fivf.nprobe = p
    rec, qps = measure(fivf)
    record("IVF", "Faiss", f"nprobe={p}", rec, qps, faiss_mb(fivf), b)
del fivf

In [ ]:
# --- 3. PQ (no cells: every query is compared with all 1M codes)
from vectorforge.pq import PQIndex

def build_pq():
    index = PQIndex(m=PQ_M)
    index.train(xt)
    index.add(xb)
    return index

pq, b = build(build_pq)
rec, qps = measure(pq)
record("PQ", "VectorForge", f"m={PQ_M}", rec, qps, (pq.codes.nbytes + pq.codebooks.nbytes) / MB, b)
del pq

def build_faiss_pq():
    index = faiss.IndexPQ(d, PQ_M, 8)
    index.train(xt)
    index.add(xb)
    return index

fpq, b = faiss_build(build_faiss_pq)
rec, qps = measure(fpq)
record("PQ", "Faiss", f"m={PQ_M}", rec, qps, faiss_mb(fpq), b)
del fpq

In [ ]:
# --- 4. IVF-PQ (with residuals)
from vectorforge.ivfpq import IVFPQIndex

def build_ivfpq():
    index = IVFPQIndex(nlist=NLIST, m=PQ_M)
    index.train(xt)
    index.add(xb)
    return index

ivfpq, b = build(build_ivfpq)
ivfpq_mb = (sum(c.nbytes for c in ivfpq.list_codes) + 8 * n
            + ivfpq.centroids.nbytes + ivfpq.pq.codebooks.nbytes) / MB    # codes + ids + centroids + codebooks
for p in NPROBES:
    ivfpq.nprobe = p
    rec, qps = measure(ivfpq)
    record("IVF-PQ", "VectorForge", f"nprobe={p}", rec, qps, ivfpq_mb, b)
del ivfpq

def build_faiss_ivfpq():
    index = faiss.IndexIVFPQ(faiss.IndexFlatL2(d), d, NLIST, PQ_M, 8)
    index.train(xt)
    index.add(xb)
    return index

fivfpq, b = faiss_build(build_faiss_ivfpq)
for p in NPROBES:
    fivfpq.nprobe = p
    rec, qps = measure(fivfpq)
    record("IVF-PQ", "Faiss", f"nprobe={p}", rec, qps, faiss_mb(fivfpq), b)
del fivfpq

In [ ]:
# --- 5. HNSW: Faiss first (fast), then yours (the long part: 1M inserts in Python)
def build_faiss_hnsw():
    index = faiss.IndexHNSWFlat(d, HNSW_M)
    index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
    index.add(xb)
    return index

fhnsw, b = faiss_build(build_faiss_hnsw)
for ef in EF_SEARCHES:
    fhnsw.hnsw.efSearch = ef
    rec, qps = measure(fhnsw)
    record("HNSW", "Faiss", f"ef_search={ef}", rec, qps, faiss_mb(fhnsw), b)
del fhnsw

from vectorforge.hnsw import HNSWIndex

hnsw = HNSWIndex(M=HNSW_M, ef_construction=HNSW_EF_CONSTRUCTION)
t0 = time.perf_counter()
CHUNK = 50_000
for s in range(0, n, CHUNK):          # add in chunks only to print progress
    hnsw.add(xb[s:s + CHUNK])
    done = s + CHUNK
    el = time.perf_counter() - t0
    print(f"  HNSW: {done:,} / {n:,} inserted, {el / 60:.1f} min so far, "
          f"about {el / done * (n - done) / 60:.0f} min left", flush=True)
b = time.perf_counter() - t0
links = sum(len(nb) for layer in hnsw.graph for nb in layer.values())
hnsw_mb = (xb.nbytes + 8 * links) / MB                           # vectors + 8 bytes per link
print(f"links per vector: {links / n:.1f}, vectors per layer: {[len(l) for l in hnsw.graph]}")
for ef in EF_SEARCHES:
    hnsw.ef_search = ef
    rec, qps = measure(hnsw)
    record("HNSW", "VectorForge", f"ef_search={ef}", rec, qps, hnsw_mb, b)

In [ ]:
# --- results: table + plot
import matplotlib.pyplot as plt

print(f"{'index':<7} {'impl':<12} {'setting':<16} {'recall@10':>9} {'QPS':>10} {'MB':>8} {'build s':>8}")
for r in rows:
    print(f"{r[0]:<7} {r[1]:<12} {r[2]:<16} {r[3]:>9.3f} {r[4]:>10,.0f} {r[5]:>8.1f} {r[6]:>8.1f}")

colors = {"IVF": "tab:blue", "PQ": "tab:purple", "IVF-PQ": "tab:green", "HNSW": "tab:orange"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for name, c in colors.items():
    for impl, style in (("VectorForge", "o-"), ("Faiss", "s--")):
        pts = [(r[3], r[4]) for r in rows if r[0] == name and r[1] == impl]
        if pts:
            rec, qps = zip(*pts)
            ax1.plot(rec, qps, style, color=c, label=f"{name} ({impl})", alpha=1 if impl == "VectorForge" else 0.6)
for impl, m in (("VectorForge", "*"), ("Faiss", "X")):
    fl = [r for r in rows if r[0] == "Flat" and r[1] == impl]
    if fl:
        ax1.plot(fl[0][3], fl[0][4], m, color="black", markersize=13, label=f"Flat ({impl})")
ax1.set_yscale("log")
ax1.set_xlabel("recall@10")
ax1.set_ylabel("queries per second (log scale)")
ax1.set_title("Speed vs recall (searches single-threaded)")
ax1.grid(True, which="both", alpha=0.3)
ax1.legend(fontsize=7, ncol=2)

mine = {}
for r in rows:
    if r[1] == "VectorForge":
        mine.setdefault(r[0], r)
        if r[3] > mine[r[0]][3]:
            mine[r[0]] = r                      # each index at its best recall
names = list(mine)
ax2.barh(names, [mine[k][5] for k in names], color=[colors.get(k, "black") for k in names])
for i, k in enumerate(names):
    ax2.text(mine[k][5] * 1.08, i, f"{mine[k][5]:,.1f} MB, best recall {mine[k][3]:.3f}", va="center", fontsize=8)
ax2.set_xscale("log")
ax2.set_xlim(right=max(mine[k][5] for k in names) * 30)
ax2.set_xlabel("index memory, MB (log scale)")
ax2.set_title("VectorForge: memory per index")
ax2.grid(True, axis="x", which="both", alpha=0.3)

fig.suptitle(f"VectorForge vs Faiss on SIFT1M ({n:,} vectors, {len(xq)} queries, k={K})")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/sift1m.png", dpi=150)
plt.show()
print(f"saved {OUT_DIR}/sift1m_results.csv and {OUT_DIR}/sift1m.png")